# 05 · Measurement, reset and teleportation / Measurement နှင့် feed-forward

**Goal:** use measurement outcomes to control later operations. About 20 minutes.
Default dynamic execution tracks exact density branches; trajectory mode samples histories instead.

Measurement ရလဒ်ကို classical bit တွင် သိမ်းပြီး နောက် gate ကို ထိန်းချုပ်နိုင်ပါသည်။


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert tuple(map(int, nexqus.__version__.split('.')[:2])) >= (0, 6), 'Install this SDK with: python -m pip install -e .'
print('NexQus SDK:', nexqus.__version__)


NexQus SDK: 0.6.0


In [2]:
q = Circuit(1).h(0).measure(0, 'm').x(0).c_if('m', 1)
result = Simulator().run(q, shots=1000, seed=42)
print('Final quantum counts:', result.counts)
print('Intermediate measurement counts:', result.classical_counts)
assert result.counts == {'0': 1000}
np.testing.assert_allclose(list(result.classical_probabilities.values()), [.5, .5], atol=1e-12)


Final quantum counts: {'0': 1000}
Intermediate measurement counts: {'0': 503, '1': 497}


In [3]:
result = Simulator().run(Circuit(2).h(0).cx(0, 1).reset(0), shots=0)
print('After resetting q0:', result.probabilities)
np.testing.assert_allclose(result.reduced_density([0]), [[1, 0], [0, 0]], atol=1e-12)
np.testing.assert_allclose(result.reduced_density([1]), np.eye(2)/2, atol=1e-12)


After resetting q0: [0.5 0.5 0.  0. ]


Reset discards q0's state and prepares |0>. Its entangled partner remains mixed.
`c_if` supports one named bit equal to 0 or 1; compound conditions are not implemented.
Overwriting a named bit discards its old value. `joint_counts` uses `classical|quantum` strings.

Reset လုပ်ခြင်းသည် entangled partner ကို pure state သို့ ပြန်မထားပါ။


In [4]:
theta, phi = .83, -.47
target = np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)])
q = Circuit(3).ry(theta, 0).phase(phi, 0).compose(algorithms.teleportation())
result = Simulator().run(q, shots=1000, seed=42)
bob = result.reduced_density([2])
fidelity = float(np.vdot(target, bob @ target).real)
print('Alice probabilities:', result.classical_probabilities)
print('Bob state:\n', bob)
print('Teleportation fidelity:', fidelity)
np.testing.assert_allclose(bob, np.outer(target, target.conj()), atol=1e-12)
assert np.isclose(fidelity, 1)


Alice probabilities: {'00': 0.25, '01': 0.25, '10': 0.25, '11': 0.25}
Bob state:
 [[0.83743788+0.00000000e+00j 0.3289581 +1.67099499e-01j]
 [0.3289581 -1.67099499e-01j 0.16256212-2.78754181e-18j]]
Teleportation fidelity: 0.9999999999999997


In [5]:
# Remove feed-forward corrections: the receiver alone is now mixed.
without_corrections = Circuit(3).ry(theta, 0).phase(phi, 0)
without_corrections.h(1).cx(1, 2).cx(0, 1).h(0).measure(0, 'z').measure(1, 'x')
uncorrected = Simulator().run(without_corrections, shots=0).reduced_density([2])
np.testing.assert_allclose(uncorrected, np.eye(2)/2, atol=1e-12)
print('Without corrections:\n', uncorrected)


Without corrections:
 [[0.5-1.3937709e-18j 0. +0.0000000e+00j]
 [0. +0.0000000e+00j 0.5-1.3937709e-18j]]


In [6]:
correlated = Circuit(2).h(0).measure(0, 'm').x(1).c_if('m', 1)
sampled = Simulator(method='trajectory').run(correlated, trajectories=512,
                                            batch_size=64, shots=1000, seed=42)
print('Joint sampled counts:', sampled.joint_counts)
assert set(sampled.joint_counts) <= {'0|00', '1|11'}
assert sum(sampled.joint_counts.values()) == 1000


Joint sampled counts: {'0|00': 461, '1|11': 539}


**Try it:** change the teleported state's angles, including a nonzero phase. Check the reduced density,
not just P(0)/P(1): probabilities alone cannot verify phase preservation.
Exact branch count can grow exponentially; use `max_branches`/memory guards or sampled trajectories.
